# Price Elasticity — Pipeline Walkthrough

> Per-product log-log demand elasticity and revenue simulation.

This notebook walks through the production pipeline using the modules in `src/`. The model is loaded from the serialized artifact produced by `python -m src.pipeline`.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src import config
pd.set_option("display.max_columns", 50)

## 1. Data

Load the versioned sample and inspect it.

In [2]:
df = pd.read_csv(config.SAMPLE_PATH)
print(df.shape)
df.head()

(23151, 4)


,product,category,price,demand
0,Boytone - 2500W 2.1-Ch. Home Theater System - ...,"speaker, portable, bluetooth",64.99,1
1,Boytone - 2500W 2.1-Ch. Home Theater System - ...,"speaker, portable, bluetooth",69.00,1
2,Boytone - 2500W 2.1-Ch. Home Theater System - ...,"speaker, portable, bluetooth",66.00,1
3,Boytone - 2500W 2.1-Ch. Home Theater System - ...,"speaker, portable, bluetooth",74.99,1
4,Boytone - 2500W 2.1-Ch. Home Theater System - ...,"speaker, portable, bluetooth",69.99,1


## 2. Preprocessing

The same transform used in training and serving.

In [3]:
from src.preprocessing import Preprocessor
qualifying, full = Preprocessor().run(df)
print("qualifying products:", qualifying[config.PRODUCT_COL].nunique())
qualifying.head()

qualifying products: 647


,product,category,price,demand
0,Boytone - 2500W 2.1-Ch. Home Theater System - ...,"speaker, portable, bluetooth",64.99,1
1,Boytone - 2500W 2.1-Ch. Home Theater System - ...,"speaker, portable, bluetooth",69.00,1
2,Boytone - 2500W 2.1-Ch. Home Theater System - ...,"speaker, portable, bluetooth",66.00,1
3,Boytone - 2500W 2.1-Ch. Home Theater System - ...,"speaker, portable, bluetooth",74.99,1
4,Boytone - 2500W 2.1-Ch. Home Theater System - ...,"speaker, portable, bluetooth",69.99,1


## 3. Model and evaluation

Metrics from the serialized model card.

In [4]:
card = json.loads(Path(config.MODEL_CARD_PATH).read_text())
print(json.dumps(card, indent=2)[:1800])

{
  "schema_version": "1.0",
  "created_at": "2026-06-14T14:55:17+00:00",
  "dataset": "datafiniti/electronic-products-prices",
  "data_sha256": "85ba4467981c3706a13eb6dccf913845653d537bc7f41cdaed78dc350b38441a",
  "problem": "per-product price elasticity (log-log demand regression)",
  "summary": {
    "total_products": 647,
    "well_fit_products": 459,
    "reliable_products": 108,
    "pct_negative_sign": 31.1,
    "median_reliable_elasticity": -2.0298,
    "pct_elastic_among_reliable": 81.5
  },
  "most_elastic": [
    {
      "product": "Seagate Backup Plus Hub 8TB External Desktop Hard Drive Storage + 2mo Adobe CC Photography (STEL8000100)",
      "elasticity": -9.9362,
      "ci95": [
        -12.1866,
        -7.6859
      ],
      "r2": 0.6878,
      "n": 36,
      "avg_price": 183.75,
      "avg_demand": 3.72,
      "category": "drive, storage, hard",
      "well_fit": true,
      "elastic": true,
      "reliable": true
    },
    {
      "product": "Details About Seagate 4 

## 4. Prediction

The serving contract on a representative input.

In [5]:
from src.predict import Predictor
pred = Predictor()
prod = pred.products()[0]
print("product:", prod)
print("elasticity:", pred.elasticity(prod)["elasticity"])
print("10% discount impact:", pred.simulate(prod, -0.10))

product: 1TB WD Black Mobile OEM Hard Drive (WD10JPLX)
elasticity: -3.683
10% discount impact: {'elasticity': -3.683, 'classification': 'elastic', 'new_price': 64.92, 'demand_change_pct': 47.4, 'revenue_change_pct': 32.7}


## Reproduce

Run the full pipeline end to end:

```
python -m src.pipeline
```